# 🎯 [MNT04_M08] 장시간 비행/기동 센서 데이터 대용량 스트림 주입 스트레스 테스트
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT04/mnt04_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT04_M08 |
| **미션 주제** | 장시간 비행/기동 센서 데이터 대용량 스트림 주입 스트레스 테스트 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
항공·기갑 장비의 엔진과 구동계는 고장이 나기 전에 진동, 온도, 압력에 미세한 변화가 먼저 나타나지만, 고정 임계값만으로는 고부하 운용일을 이상으로 오인하거나 초기 열화를 놓쳐 수리부속을 제때 확보하지 못한다. 본 프로젝트는 고장 실적과 수리부속 소모를 연결해 정비와 부품 관리의 연계 문제를 학습하는 것을 목표로, 교육용 가상 기동체 48대의 센서 시계열과 부품 소모를 코드로 만들어 분석한다. 센서 데이터셋 정의, 잡음 제거와 FFT 특징량 추출, 임계값 Baseline과 부품 소모 파레토 분석, 오토인코더 기반 미세 변화 감지와 파생 피처 모델, 잔존수명(RUL) 예측과 이상 신뢰도 점수, 설명가능한 경보 화면, 검사관 승인(HITL) 기반 조기 입고 시뮬레이터, 대용량 스트림 스트레스 시험, AAR 오경보 분석과 임계 재보정, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 고장 전에 이상 징후를 일찍 잡아 정비 시점과 수리부속 소요를 예고하되, 조기 입고 결정은 검사관이 근거를 보고 내리도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전(정비대대, 군수지원사령부)에 즉시 도입 가능한 "완성형 국방 예지정비(PdM) 및 PBL 품목관리 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt04_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt04_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **센서 데이터 정제 및 전처리 모듈** | 고주파 센서 신호 잡음 제거, 결측치 보정 및 FFT 특징 추출 파이프라인(`src/etl/mnt04_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 3-시그마 임계값 Baseline 및 부품 소모 파레토 분석(`notebooks/mnt04_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 오토인코더 기반 미세 진동 변화 감지 및 파생 피처 엔지니어링 모듈(`src/features/mnt04_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 잔존수명(RUL) 엔진** | 부품 RUL 예측 앙상블 모델 및 종합 상태 이상 신뢰도 엔진(`src/models/mnt04_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 오토인코더 재구성 오차 분해 및 XAI 원인 설명기(`src/explainability/mnt04_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 조기 입고 시뮬레이터** | 검사관 HITL 승인 큐, 감사 추적 및 조기 정비 시뮬레이터(`src/optimization/mnt04_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트림 주입 & 관제 대시보드** | 대용량 센서 스트림 주입 스트레스 시험 및 실시간 관제 대시보드(`src/dashboard/mnt04_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 오경보 원인 분석 및 임계 파라미터 재보정 ROI 보고서(`reports/mnt04_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 데이터 기반 챔피언-도전자 자동 재학습 및 E2E 통합 파이프라인(`pipeline/mnt04_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 장시간 비행/기동 센서 데이터 대용량 스트림 주입 스트레스 테스트
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

장시간 기동 센서 데이터 대용량 스트림 주입 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **장시간 기동 센서 데이터를 대용량 스트림으로 주입해 처리 시간, 경보 정확도, 불량 패킷 격리를 확인하는 스트레스 시험을 수행한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 센서 측정 1,000건에 결측 패킷, 센서 고정, 포화, 범위 이탈을 섞어 한꺼번에 주입하고 정상 처리·격리·유실 건수와 처리 시간을 센다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/dashboard/mnt04_app.py (Streamlit/Dash 등)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 의사결정 지원에 최적화된 시각적 위계 및 사용자 경험(UX)

#### 미션 PASS 조건

- 정량 기준: 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상


- 진행 기준: 웹 기반 분석 대시보드 구동 및 실시간 필터링 기능 검증


- 검증 방법: 실제 군수 데이터 연동 화면 테스트 및 UI 동작 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 08 산출물 파일 연계]: src/dashboard/mnt04_app.py
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile src/dashboard/mnt04_app.py
# ============================================================================
"""장시간 기동 센서 데이터를 1,000~100,000건 규모로 한꺼번에 주입해 검증·특징 추출·오토인코더 점수·경보 판정을 배치로 처리하는 스트림 시험을 수행하고, 처리 시간과 상태 화면 구성 시간, 경보 정확도, 불량 패킷(결측·고정값·포화·중복) 격리와 유실 0건 여부를 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 기동체 48대의 엔진·구동계 진동·온도·압력 센서 자료 생성기 (실제 장비·부대·정비 실적 없음)
# 개체마다 정상 -> 열화 -> 고장(수명종료)까지 하루 1회 측정(진동 파형 256표본 + 온도 + 압력 + 부하)을 만든다.
# 열화 유형 4가지: bearing(베어링), gear(기어), oil_pump(오일 펌프), cooling(냉각 팬 모터)
# ---------------------------------------------------------------------------
FS, NS = 1024, 256                              # 가상 표본화 주파수(Hz), 파형 1개당 표본 수
TT = np.arange(NS) / FS
MODES = {  # 열화 유형: (발생 확률, 평균 수명(일), 교체 수리부속, 가상 단가지수)
    "bearing": (0.40, 160, "주 베어링 세트", 6.0), "gear": (0.25, 175, "구동 기어 세트", 9.0),
    "oil_pump": (0.20, 150, "오일 펌프", 4.0), "cooling": (0.15, 140, "냉각 팬 모터", 3.0)}
KIT_COST = 1.0                                  # 모든 정비에 함께 소모되는 씰·필터 키트 단가지수
BANDS = {"b50": (40, 60), "b100": (90, 110), "b150": (140, 156), "b165": (156, 176), "b200": (190, 215)}


def make_runs(n=48, seed=1):
    """반환: units(개체 정보), snap(일별 측정 요약), W(진동 파형 [측정 수, 256])"""
    rng = np.random.default_rng(seed)
    names = list(MODES)
    mode = rng.choice(names, n, p=[MODES[m][0] for m in names])
    life = np.clip(np.array([rng.normal(MODES[m][1], 18) for m in mode]), 90, 230).astype(int)
    start = np.sort(rng.integers(0, 240, n))
    units = pd.DataFrame({"unit": [f"U{i:02d}" for i in range(n)], "mode": mode, "life": life, "start": start, "base_load": rng.uniform(0.4, 0.75, n)})
    rows = []
    for u in units.itertuples():
        d = np.arange(u.life + 1)
        onset = 0.45 * u.life
        deg = np.clip((d - onset) / (u.life - onset), 0, 1) ** 1.5
        load = np.clip(u.base_load + rng.normal(0, 0.06, len(d)), 0.2, 1.0)
        hi = rng.random(len(d)) < 0.07                                            # 고부하 운용일(정상이지만 진동·온도 상승)
        load = np.where(hi, rng.uniform(0.95, 1.1, len(d)), load)
        rows.append(pd.DataFrame({"unit": u.unit, "day": d, "cal": u.start + d, "rul": u.life - d, "deg": deg, "load": load, "hi_load": hi, "mode": u.mode}))
    snap = pd.concat(rows, ignore_index=True)
    S = len(snap)
    snap["ambient"] = 15 + 12 * np.sin(2 * np.pi * snap["cal"] / 365) + rng.normal(0, 2, S)
    dg, ld, md = snap["deg"].values, snap["load"].values, snap["mode"].values
    isb, isg, iso, isc = (md == "bearing"), (md == "gear"), (md == "oil_pump"), (md == "cooling")
    snap["temp"] = 55 + 22 * ld + 0.5 * snap["ambient"] + 9 * dg * iso + 26 * dg * isc + 3 * dg * (isb | isg) + rng.normal(0, 1.5, S)
    snap["press"] = 450 - 12 * ld - 85 * dg * iso - 15 * dg * isc + rng.normal(0, 6, S)
    ph = rng.uniform(0, 2 * np.pi, (S, 3))
    a1, a2 = 1.0 * (0.5 + ld), 0.4 * (0.5 + ld)
    W = (a1[:, None] * np.sin(2 * np.pi * 50 * TT + ph[:, :1]) + a2[:, None] * np.sin(2 * np.pi * 100 * TT + ph[:, 1:2])
         + rng.normal(0, 0.35, (S, NS)))
    W += (0.9 * dg * isb)[:, None] * np.sin(2 * np.pi * 165 * TT + ph[:, 2:3])
    W += (0.8 * dg * isg)[:, None] * (np.sin(2 * np.pi * 150 * TT + ph[:, 2:3]) + 0.8 * np.sin(2 * np.pi * 100 * TT))
    W += (0.5 * dg * iso)[:, None] * np.sin(2 * np.pi * 200 * TT + ph[:, 2:3])
    k = rng.poisson(10 * dg * isb)                                                 # 베어링 열화: 충격성 펄스(첨도 상승)
    pos, amp = rng.integers(0, NS, (S, 30)), rng.normal(0, 3.0, (S, 30)) * dg[:, None]
    amp = amp * (np.arange(30)[None, :] < k[:, None])
    np.add.at(W, (np.repeat(np.arange(S), 30), pos.ravel()), amp.ravel())
    return units, snap, W


def extract(W):
    """진동 파형 -> 특징량: RMS, 첨도, 파고율, 최대-최소, FFT 대역 에너지(50/100/150/165/200Hz 부근)"""
    x = W - W.mean(axis=1, keepdims=True)
    rms = np.sqrt((x ** 2).mean(axis=1))
    out = {"rms": rms, "kurt": kurtosis(x, axis=1, fisher=False), "crest": np.abs(x).max(axis=1) / rms, "p2p": x.max(axis=1) - x.min(axis=1)}
    A = np.abs(np.fft.rfft(x * np.hanning(NS), axis=1)) / NS
    f = np.fft.rfftfreq(NS, 1 / FS)
    for nm, (lo, hi) in BANDS.items():
        out[nm] = np.sqrt((A[:, (f >= lo) & (f < hi)] ** 2).sum(axis=1))
    return pd.DataFrame(out)

# ---------------------------------------------------------------------------
# 공통 분석 조각: 일별 특징량 표(파형 특징 + 온도·압력 + 파생), 개체 단위 학습/검증 분할(앞선 24대 학습, 뒤 24대 검증)
# ---------------------------------------------------------------------------
BASE = ["rms", "kurt", "crest", "p2p", "b50", "b100", "b150", "b165", "b200", "temp", "press"]
ROLL = ["rms", "kurt", "b150", "b165", "b200", "temp", "press"]


def build_features(snap, W):
    D = pd.concat([snap.reset_index(drop=True), extract(W)], axis=1)
    D["temp_res"] = D["temp"] - (55 + 22 * D["load"] + 0.5 * D["ambient"])        # 부하·외기 대비 온도 잔차
    D["press_res"] = D["press"] - (450 - 12 * D["load"])                           # 부하 대비 압력 잔차
    D["rms_n"] = D["rms"] / (0.5 + D["load"])                                      # 부하 정규화 RMS
    D["hf_ratio"] = (D["b150"] + D["b165"] + D["b200"]) / (D["b50"] + 1e-6)         # 고주파/기본주파 비
    g = D.groupby("unit")
    for c in ROLL:
        D[c + "_m3"] = g[c].transform(lambda s: s.rolling(3, min_periods=1).mean())
    for c in ("rms_n", "temp_res", "press_res"):
        D[c + "_d"] = g[c].diff().fillna(0)
    return D


DERIVED = ["temp_res", "press_res", "rms_n", "hf_ratio"] + [c + "_m3" for c in ROLL] + ["rms_n_d", "temp_res_d", "press_res_d"]
FEATS = BASE + DERIVED
AE_COLS = ["rms_n", "kurt", "b150", "b165", "b200", "hf_ratio", "temp_res", "press_res"]


def split_units(units, n_train=24):
    return list(units["unit"][:n_train]), list(units["unit"][n_train:])


class AE:
    """오토인코더(입력 8 -> 10 -> 4 -> 10 -> 8): 정상 자료만 재구성하도록 학습하고, 재구성 오차를 이상 점수로 쓴다."""

    def __init__(self, cols=AE_COLS, q=0.99, seed=0):
        self.cols, self.q, self.seed = cols, q, seed

    def fit(self, healthy):
        self.sc = StandardScaler().fit(healthy[self.cols])
        Z = self.sc.transform(healthy[self.cols])
        self.net = MLPRegressor(hidden_layer_sizes=(10, 4, 10), max_iter=800, random_state=self.seed).fit(Z, Z)
        self.thr = float(np.quantile(self.err(healthy), self.q))
        return self

    def contrib(self, d):
        Z = self.sc.transform(d[self.cols])
        return (Z - self.net.predict(Z)) ** 2                 # 입력 항목별 재구성 오차(이상 기여도)

    def err(self, d):
        return self.contrib(d).mean(axis=1)


def latch(d, e, thr, k=2):
    """같은 개체에서 이상 점수가 k일 연속 기준을 넘으면 경보를 켜고 유지(래치)한다."""
    out, e = np.zeros(len(d), bool), np.asarray(e)
    for idx in d.reset_index(drop=True).groupby("unit").indices.values():
        run, on = 0, False
        for i in idx:
            run = run + 1 if e[i] > thr else 0
            on = on or run >= k
            out[i] = on
    return out

units, snap, W = make_runs()
D = build_features(snap, W)
tr_u, va_u = split_units(units)
tr, va = D[D["unit"].isin(tr_u)].reset_index(drop=True), D[D["unit"].isin(va_u)].reset_index(drop=True)
Wva = W[D["unit"].isin(va_u).values]
ae = AE().fit(tr[tr["deg"] == 0])
KEEP = ["unit", "day", "load", "ambient", "temp", "press"]


def score_batch(w, meta):
    """스트림 한 묶음: 파형 -> 특징 -> 이상 점수(배수). 점수는 개체 연속성(래치)과 무관한 단건 값."""
    f = extract(w)
    d = pd.DataFrame({"rms_n": f["rms"] / (0.5 + meta["load"]), "kurt": f["kurt"], "b150": f["b150"], "b165": f["b165"], "b200": f["b200"],
                      "hf_ratio": (f["b150"] + f["b165"] + f["b200"]) / (f["b50"] + 1e-6),
                      "temp_res": meta["temp"] - (55 + 22 * meta["load"] + 0.5 * meta["ambient"]), "press_res": meta["press"] - (450 - 12 * meta["load"])})
    return ae.err(d) / ae.thr


def valid(w, meta):
    """불량 패킷 판정: 결측(NaN), 센서 고정(분산 0), 포화(|값|>12), 값 범위 이탈."""
    bad = np.isnan(w).any(axis=1) | (np.nanstd(w, axis=1) < 1e-6) | (np.abs(np.nan_to_num(w)).max(axis=1) > 12)
    bad |= ~meta["temp"].between(20, 160).values | ~meta["press"].between(100, 600).values
    return ~bad


def make_stream(n, bad_rate, seed=0):
    rng = np.random.default_rng(seed)
    pick = rng.integers(0, len(va), n)
    w = Wva[pick] + rng.normal(0, 0.02, (n, NS))
    meta = va.iloc[pick][KEEP + ["deg"]].reset_index(drop=True)
    kind = np.array([""] * n, dtype=object)
    bad = rng.random(n) < bad_rate
    k = rng.integers(0, 4, n)
    for t, nm in enumerate(["결측 패킷", "센서 고정", "포화", "범위 이탈"]):
        m = bad & (k == t)
        kind[m] = nm
        if t == 0: w[m, 10:40] = np.nan
        if t == 1: w[m] = 0.5
        if t == 2: w[m, ::7] = 50.0
        if t == 3: meta.loc[m, "temp"] = 400.0
    return w, meta, kind


# 1) 1,000건 동시 주입
t0 = time.perf_counter()
w, meta, kind = make_stream(1000, 0.06)
ok = valid(w, meta)
s = np.full(1000, np.nan)
s[ok] = score_batch(w[ok], meta[ok].reset_index(drop=True))
sec = time.perf_counter() - t0
inj = int((kind != "").sum())
print(f"[교육용가상자료] 1,000건 동시 주입: 정상 처리 {int(ok.sum())}건, 격리 {int((~ok).sum())}건, 유실 {1000 - int(ok.sum()) - int((~ok).sum())}건, 처리 시간 {sec:.2f}초 (목표 2초 이내: {'달성' if sec <= 2 else '미달'})")
print(f"  불량 주입 {inj}건 중 격리 {int(((kind != '') & ~ok).sum())}건 (" + ", ".join(f"{k} {int((kind == k).sum())}" for k in ["결측 패킷", "센서 고정", "포화", "범위 이탈"]) + ")")

# 2) 경보 정확도: 정상 구간과 열화가 뚜렷한 구간(deg>=0.3)을 판정 대상으로 하고, 열화 초기 회색 구간은 정확도에서 제외하고 따로 표시
al = s > 1.0
y = (meta["deg"] >= 0.3).values
eval_m = ok & ((meta["deg"] == 0).values | y)
acc = (al[eval_m] == y[eval_m]).mean()
tp, fp, fn = (al & y & ok).sum(), (al & ~y & eval_m).sum(), (~al & y & ok).sum()
print(f"경보 정확도 {acc:.1%} (목표 95% 이상: {'달성' if acc >= 0.95 else '미달'}), 재현율 {tp / (tp + fn):.1%}, 정밀도 {tp / (tp + fp):.1%}, 경보 {int((al & ok).sum())}건")
gray = ok & (meta["deg"] > 0).values & (meta["deg"] < 0.3).values
print(f"  참고: 열화 초기(0~0.3) 구간 {int(gray.sum())}건 중 경보 {al[gray].mean():.0%}(회색 구간은 정확도에서 제외)")

# 3) 상태 화면 구성: 최근 30일 x 24대 요약(개체별 최댓값·경보 여부) 집계 시간
t0 = time.perf_counter()
last = va[va["day"] >= va.groupby("unit")["day"].transform("max") - 30]
sc = score_batch(Wva[last.index.values], last.reset_index(drop=True))
view = last.assign(score=sc).groupby("unit").agg(max_score=("score", "max"), last_score=("score", "last"))
view["상태"] = np.where(view["last_score"] > 1, "경보", np.where(view["max_score"] > 1, "주의", "정상"))
tv = time.perf_counter() - t0
print(f"상태 화면 구성(24대 x 최근 30일 {len(last)}건) {tv:.2f}초 (목표 2초 이내: {'달성' if tv <= 2 else '미달'}), " + ", ".join(f"{k} {v}대" for k, v in view["상태"].value_counts().items()))

# 4) 규모 확대: 배치 처리량과 메모리 한계 대응(배치 크기 5,000)
print("\n[주입 규모별 처리] 건수 | 시간 | 처리량 | 격리 | 유실")
sizes = (1000, 10000, 50000, 100000)
ts = []
for n in sizes:
    w, meta, kind = make_stream(n, 0.06, seed=n)
    t0 = time.perf_counter(); done = quarantine = 0
    for i in range(0, n, 5000):
        wb, mb = w[i:i + 5000], meta.iloc[i:i + 5000].reset_index(drop=True)
        okb = valid(wb, mb)
        if okb.any():
            score_batch(wb[okb], mb[okb].reset_index(drop=True))
        done += int(okb.sum()); quarantine += int((~okb).sum())
    t = time.perf_counter() - t0; ts.append(t)
    print(f" {n:7d}  {t:5.2f}초  {n / t:8.0f}건/초  {quarantine:5d}  {n - done - quarantine}")
print("운용 기준: 전체 격리율이 20%를 넘으면 센서 계통 점검 알림, 처리 지연 시 FFT 대역 5개만 계산하는 축소 모드로 전환")

plt.figure(figsize=(6, 4))
plt.plot(sizes, ts, marker="o"); plt.xscale("log"); plt.xlabel("stream size"); plt.ylabel("seconds"); plt.title("Stream stress test")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 1,000건 동시 주입: 정상 처리 945건, 격리 55건, 유실 0건, 처리 시간 0.03초 (목표 2초 이내: 달성)
  불량 주입 55건 중 격리 55건 (결측 패킷 15, 센서 고정 16, 포화 15, 범위 이탈 9)
경보 정확도 98.0% (목표 95% 이상: 달성), 재현율 98.0%, 정밀도 97.3%, 경보 330건
  참고: 열화 초기(0~0.3) 구간 231건 중 경보 15%(회색 구간은 정확도에서 제외)
상태 화면 구성(24대 x 최근 30일 744건) 0.02초 (목표 2초 이내: 달성), 경보 24대

[주입 규모별 처리] 건수 | 시간 | 처리량 | 격리 | 유실
    1000   0.02초     48180건/초     68  0
   10000   0.21초     48054건/초    618  0
   50000   1.23초     40710건/초   2951  0
  100000   3.51초     28472건/초   5940  0
운용 기준: 전체 격리율이 20%를 넘으면 센서 계통 점검 알림, 처리 지연 시 FFT 대역 5개만 계산하는 축소 모드로 전환
```
</details>
